# Qwen GPU memory tests: 4096, then 6144

Runs the patched answer-only loss on the connected GPU. Each test includes the longest fitting training question and all four rotations. No full training run is launched. The second test starts only if the first completes training, adapter reload and evaluation. Results and console logs remain on the runtime disk.


In [ ]:
"""Find and read the three training CSVs without a browser upload widget."""

import csv
from pathlib import Path

REQUIRED_FILES = ("train.csv", "validation.csv", "TestData.csv")


def find_data_dir(data_dir=None, start=None):
    """Find nearby local files; cloud runtimes still require transferred inputs."""
    if data_dir is not None:
        folder = Path(data_dir).expanduser().resolve()
        if not all((folder / name).is_file() for name in REQUIRED_FILES):
            raise FileNotFoundError(f"Expected {', '.join(REQUIRED_FILES)} in {folder}")
        return folder
    # Explicit cloud locations also work when the notebook working directory differs.
    for folder in [Path("/content"), Path("/content/data")]:
        if all((folder / name).is_file() for name in REQUIRED_FILES):
            return folder
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.is_dir():
        matches = sorted({p.parent for p in kaggle_root.rglob("train.csv")
                          if all((p.parent / name).is_file() for name in REQUIRED_FILES)})
        if len(matches) > 1:
            raise ValueError("Multiple Kaggle datasets match; set DATA_DIR explicitly.")
        if matches:
            return matches[0]
    start = Path(start or Path.cwd()).resolve()
    for base in [start, *list(start.parents)[:3]]:
        for folder in [base / "kaggle-bundle" / "data", base / "data", base]:
            if all((folder / name).is_file() for name in REQUIRED_FILES):
                return folder
    return None


def load_csv_data(data_dir=None):
    folder = find_data_dir(data_dir)
    if folder is None:
        raise FileNotFoundError("CSV folder not found. Pass data_dir or run from the repository/bundle folder.")
    datasets = {}
    required = {"class", "Question", "a", "b", "c", "d", "Answer"}
    for name in REQUIRED_FILES:
        with (folder / name).open(encoding="utf-8-sig", newline="") as handle:
            reader = csv.DictReader(handle)
            if not required.issubset(reader.fieldnames or []):
                raise ValueError(f"Unexpected columns in {name}")
            rows = list(reader)
        if not rows:
            raise ValueError(f"Empty CSV: {name}")
        datasets[name] = rows
    return datasets



import json,sys,subprocess,importlib.metadata
from datetime import datetime,timezone
import torch
DATA_DIR=find_data_dir()
if DATA_DIR is None:
    raise FileNotFoundError('Put train.csv, validation.csv and TestData.csv in /content or /content/data first.')
if not torch.cuda.is_available():
    raise RuntimeError('Connect this notebook to your Colab GPU kernel.')
free,total=torch.cuda.mem_get_info()
print('Data:',DATA_DIR)
print('GPU:',torch.cuda.get_device_name(0),'free GiB:',round(free/1024**3,2),'total GiB:',round(total/1024**3,2))
print('Versions:',{p:importlib.metadata.version(p) for p in ['torch','transformers','peft','bitsandbytes','accelerate']})
if free < min(12*1024**3,0.8*total):
    raise RuntimeError('GPU already occupied. Stop other training processes or restart this kernel before testing.')
TEST_ROOT=Path.cwd()/'memory-smoke-results'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
TEST_ROOT.mkdir(parents=True,exist_ok=False)
print('Results:',TEST_ROOT)


In [ ]:
# Exact current script: benchmark.py
Path('benchmark.py').write_text('"""Zero-shot MCQ baseline. No training or training-data access."""\n\nimport argparse\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport re\nimport time\n\n\ndef build_prompt(row):\n    choices = "\\n".join(f"{key}) {row[key]}" for key in "abcd")\n    return (\n        "Answer the following multiple-choice question about a Unity C# game project.\\n"\n        "Choose the single best answer. End your response with Answer:a, Answer:b, "\n        "Answer:c, or Answer:d.\\n\\n"\n        f"{row[\'Question\']}\\n{choices}"\n    )\n\n\ndef parse_answer(response):\n    # An unfinished thinking block is not a final answer.\n    if "<think>" in response and "</think>" not in response:\n        return None\n    final = response.rsplit("</think>", 1)[-1].strip()\n    match = re.search(r"Answer\\s*:\\s*([abcd])\\s*[.!]?\\s*$", final, re.I)\n    if match:\n        return match.group(1).lower()\n    return final.lower() if re.fullmatch(r"[abcd]", final, re.I) else None\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--model", default="Qwen/Qwen3-4B")\n    parser.add_argument("--data", type=Path, default=Path("TestData.csv"))\n    parser.add_argument("--output", type=Path, required=True,\n                        help="New directory for this run; existing directories are rejected.")\n    parser.add_argument("--limit", type=int, help="First N rows for smoke checks only")\n    parser.add_argument("--thinking", action="store_true", help="Qwen3 thinking mode")\n    parser.add_argument("--max-new-tokens", type=int)\n    parser.add_argument("--four-bit", action="store_true")\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args(argv)\n    if args.limit is not None and args.limit < 1:\n        parser.error("--limit must be positive")\n    if args.max_new_tokens is not None and args.max_new_tokens < 1:\n        parser.error("--max-new-tokens must be positive")\n    if args.thinking and not args.model.startswith("Qwen/Qwen3-"):\n        parser.error("--thinking is implemented only for Qwen3 models")\n    with args.data.open(encoding="utf-8-sig", newline="") as handle:\n        reader = csv.DictReader(handle)\n        required = {"class", "Question", "a", "b", "c", "d", "Answer"}\n        if not required.issubset(reader.fieldnames or []):\n            parser.error("Unexpected test CSV schema")\n        rows = list(reader)\n    if args.limit:\n        rows = rows[:args.limit]\n    if not rows or any(row["Answer"].strip().lower() not in "abcd" or\n                       len(row["Answer"].strip()) != 1 for row in rows):\n        parser.error("Empty dataset or invalid answer labels")\n\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("A CUDA GPU is required. Select a GPU runtime in Colab/Kaggle.")\n    args.output.mkdir(parents=True, exist_ok=False)\n    set_seed(args.seed)\n    tokenizer = AutoTokenizer.from_pretrained(args.model)\n    config = {"device_map": "auto", "torch_dtype": torch.float16}\n    if args.four_bit:\n        config["quantization_config"] = BitsAndBytesConfig(\n            load_in_4bit=True, bnb_4bit_quant_type="nf4",\n            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,\n        )\n    model = AutoModelForCausalLM.from_pretrained(args.model, **config).eval()\n    max_tokens = args.max_new_tokens or (2048 if args.thinking else 128)\n    generation = {"max_new_tokens": max_tokens, "do_sample": args.thinking,\n                  "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id}\n    if args.thinking:\n        generation.update(temperature=0.6, top_p=0.95, top_k=20)\n    metadata = {\n        "model": args.model, "model_commit": getattr(model.config, "_commit_hash", None),\n        "dataset_sha256": hashlib.sha256(args.data.read_bytes()).hexdigest(),\n        "rows": len(rows), "smoke_test": args.limit is not None, "seed": args.seed,\n        "thinking": args.thinking, "four_bit": args.four_bit, "generation": generation,\n        "gpu": torch.cuda.get_device_name(0), "prompt_example": build_prompt(rows[0]),\n        "versions": {p: importlib.metadata.version(p) for p in\n                     ["torch", "transformers", "accelerate", "bitsandbytes"]},\n        "protocol": "Zero-shot; no training examples or source code; invalid answers count as wrong.",\n    }\n    (args.output / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")\n    predictions = []\n    start = time.perf_counter()\n    with (args.output / "predictions.jsonl").open("x", encoding="utf-8") as handle:\n        for index, row in enumerate(rows):\n            messages = [{"role": "user", "content": build_prompt(row)}]\n            template_args = {}\n            if args.model.startswith("Qwen/Qwen3-"):\n                template_args["enable_thinking"] = args.thinking\n            text = tokenizer.apply_chat_template(\n                messages, tokenize=False, add_generation_prompt=True, **template_args)\n            inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)\n            tick = time.perf_counter()\n            with torch.inference_mode():\n                output = model.generate(**inputs, **generation)\n            generated = output[0, inputs.input_ids.shape[1]:]\n            response = tokenizer.decode(generated, skip_special_tokens=True)\n            answer = parse_answer(response)\n            result = {\n                "row": index, "class": row["class"], "question": row["Question"],\n                "response": response, "prediction": answer,\n                "truth": row["Answer"].strip().lower(),\n                "correct": answer == row["Answer"].strip().lower(),\n                "seconds": time.perf_counter() - tick,\n                "generated_tokens": len(generated),\n                "token_budget_reached": len(generated) >= max_tokens,\n            }\n            predictions.append(result)\n            handle.write(json.dumps(result, ensure_ascii=False) + "\\n")\n            handle.flush()\n            print(f"{index + 1}/{len(rows)} correct={sum(p[\'correct\'] for p in predictions)}", flush=True)\n    correct = sum(p["correct"] for p in predictions)\n    labels = {key: sum(p["truth"] == key for p in predictions) for key in "abcd"}\n    summary = {\n        "model": args.model, "questions": len(rows), "correct": correct,\n        "accuracy": correct / len(rows),\n        "invalid_answers": sum(p["prediction"] is None for p in predictions),\n        "token_budget_reached": sum(p["token_budget_reached"] for p in predictions),\n        "uniform_random_expected_accuracy": 0.25,\n        "label_counts": labels, "constant_choice_accuracy": {k: v / len(rows) for k, v in labels.items()},\n        "elapsed_seconds": time.perf_counter() - start,\n        "smoke_test": args.limit is not None,\n    }\n    (args.output / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")\n    print(json.dumps(summary, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n',encoding='utf-8')


In [ ]:
# Exact current script: finetune_qwen_coder.py
Path('finetune_qwen_coder.py').write_text('"""One-epoch QLoRA experiment; test labels never enter training or prompts."""\n\nimport argparse\nfrom collections import defaultdict\nimport csv\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport re\nimport time\n\nos.environ.setdefault("USE_TF", "0")\nos.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\n\nfrom benchmark import build_prompt, parse_answer\n\nMODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\\n", encoding="utf-8")\n\n\ndef read_csv(path, required):\n    with Path(path).open(encoding="utf-8-sig", newline="") as handle:\n        reader = csv.DictReader(handle)\n        if not set(required).issubset(reader.fieldnames or []):\n            raise ValueError(f"Unexpected CSV schema: {path}")\n        rows = list(reader)\n    if not rows or any(not str(row.get(key) or "").strip() for row in rows for key in required):\n        raise ValueError(f"Empty rows or required fields: {path}")\n    return rows\n\n\ndef split_training_rows(rows, seed=42, validation_fraction=0.1):\n    # Keep identical questions together even if class names or answers differ.\n    groups = defaultdict(list)\n    for index, row in enumerate(rows):\n        question = re.sub(r"\\s+", " ", row["Q"].strip().casefold())\n        groups[question].append(index)\n    keys = sorted(groups)\n    random.Random(seed).shuffle(keys)\n    validation_keys = set(keys[:max(1, round(len(keys) * validation_fraction))])\n    validation_indices = sorted(i for key in validation_keys for i in groups[key])\n    validation_set = set(validation_indices)\n    training_indices = [i for i in range(len(rows)) if i not in validation_set]\n    if not training_indices or not validation_indices:\n        raise ValueError("Need enough distinct questions for train/validation splitting")\n    return training_indices, validation_indices\n\n\ndef tokenize_example(row, tokenizer, max_length):\n    user = f"In this Unity game project, regarding {row[\'class\'].strip()}:\\n{row[\'Q\'].strip()}"\n    messages = [{"role": "user", "content": user}]\n    prefix = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)\n    full = tokenizer.apply_chat_template(\n        messages + [{"role": "assistant", "content": row["A"].strip()}],\n        tokenize=True, add_generation_prompt=False,\n    )\n    if full[:len(prefix)] != prefix:\n        raise ValueError("Chat-template prefix mismatch; refusing incorrect loss masking")\n    if len(full) > max_length:\n        return None, len(full)\n    labels = [-100] * len(prefix) + full[len(prefix):]\n    if not any(label != -100 for label in labels):\n        raise ValueError("Example contains no answer tokens")\n    return {"input_ids": full, "attention_mask": [1] * len(full), "labels": labels}, len(full)\n\n\ndef collate_examples(examples, pad_token_id):\n    import torch\n    width = math.ceil(max(len(row["input_ids"]) for row in examples) / 8) * 8\n    batch = {key: [] for key in ["input_ids", "attention_mask", "labels"]}\n    for row in examples:\n        padding = width - len(row["input_ids"])\n        for key, pad in [("input_ids", pad_token_id), ("attention_mask", 0), ("labels", -100)]:\n            batch[key].append(row[key] + [pad] * padding)\n    return {key: torch.tensor(value, dtype=torch.long) for key, value in batch.items()}\n\n\ndef evaluate_mcq(model, tokenizer, rows, output):\n    import torch\n    output.mkdir(parents=True, exist_ok=False)\n    model.eval()\n    model.config.use_cache = True\n    predictions = []\n    torch.cuda.synchronize()\n    start = time.perf_counter()\n    with (output / "predictions.jsonl").open("x", encoding="utf-8") as handle:\n        for index, row in enumerate(rows):\n            prompt = build_prompt(row)\n            text = tokenizer.apply_chat_template(\n                [{"role": "user", "content": prompt}],\n                tokenize=False, add_generation_prompt=True,\n            )\n            inputs = tokenizer(text, add_special_tokens=False, return_tensors="pt").to(model.device)\n            tick = time.perf_counter()\n            with torch.inference_mode():\n                generated = model.generate(\n                    **inputs, do_sample=False, max_new_tokens=128,\n                    pad_token_id=tokenizer.pad_token_id,\n                )[0, inputs.input_ids.shape[1]:]\n            response = tokenizer.decode(generated, skip_special_tokens=True)\n            answer = parse_answer(response)\n            result = {\n                "row": index, "class": row["class"], "question": row["Question"],\n                "response": response, "prediction": answer, "truth": row["Answer"].strip().lower(),\n                "correct": answer == row["Answer"].strip().lower(),\n                "generated_tokens": len(generated), "token_budget_reached": len(generated) >= 128,\n                "seconds": time.perf_counter() - tick,\n            }\n            predictions.append(result)\n            handle.write(json.dumps(result, ensure_ascii=False) + "\\n")\n            handle.flush()\n            if (index + 1) % 10 == 0 or index + 1 == len(rows):\n                print(f"{output.name}: {index + 1}/{len(rows)}", flush=True)\n    summary = {\n        "model": MODEL_ID, "questions": len(rows),\n        "correct": sum(row["correct"] for row in predictions),\n        "accuracy": sum(row["correct"] for row in predictions) / len(rows),\n        "invalid_answers": sum(row["prediction"] is None for row in predictions),\n        "token_budget_reached": sum(row["token_budget_reached"] for row in predictions),\n        "elapsed_seconds": time.perf_counter() - start,\n    }\n    write_json(output / "summary.json", summary)\n    with (output / "predictions.csv").open("x", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(predictions[0]))\n        writer.writeheader()\n        writer.writerows(predictions)\n    return summary, predictions\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--train", type=Path, default=Path("TrainData.csv"))\n    parser.add_argument("--test", type=Path, default=Path("TestData.csv"))\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--max-length", type=int, default=2048)\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n    if args.max_length < 64:\n        parser.error("--max-length must be at least 64")\n    train_rows = read_csv(args.train, ["class", "Q", "A"])\n    test_rows = read_csv(args.test, ["class", "Question", "a", "b", "c", "d", "Answer"])\n    if any(row["Answer"].strip().lower() not in list("abcd") for row in test_rows):\n        raise ValueError("Invalid MCQ answer label")\n\n    import torch\n    from functools import partial\n    from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training\n    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments, set_seed\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("Enable a CUDA GPU before running this experiment")\n    args.output.mkdir(parents=True, exist_ok=False)\n    set_seed(args.seed)\n    training_indices, validation_indices = split_training_rows(train_rows, args.seed)\n    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)\n    if tokenizer.pad_token_id is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    tokenizer.padding_side = "right"\n    prepared = {}\n    lengths = []\n    exclusions = []\n    for index, row in enumerate(train_rows):\n        example, length = tokenize_example(row, tokenizer, args.max_length)\n        lengths.append(length)\n        if example is None:\n            exclusions.append({"row": index, "class": row["class"], "tokens": length,\n                               "reason": "Exceeds max_length; excluded without truncating answers"})\n        else:\n            prepared[index] = example\n    training_indices = [i for i in training_indices if i in prepared]\n    validation_indices = [i for i in validation_indices if i in prepared]\n    if not training_indices or not validation_indices:\n        raise ValueError("No usable training or validation examples after length checks")\n    write_json(args.output / "split.json", {\n        "seed": args.seed, "train_indices": training_indices,\n        "validation_indices": validation_indices, "excluded": exclusions,\n        "note": "Grouped by identical normalized question. Semantic paraphrases may still overlap.",\n    })\n    quantization = BitsAndBytesConfig(\n        load_in_4bit=True, bnb_4bit_quant_type="nf4",\n        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,\n    )\n    model = AutoModelForCausalLM.from_pretrained(\n        MODEL_ID, quantization_config=quantization, dtype=torch.float16,\n        device_map={"": 0}, attn_implementation="sdpa",\n    )\n    revision = getattr(model.config, "_commit_hash", None)\n    training_settings = {\n        "num_train_epochs": 1, "learning_rate": 1e-4, "batch_size": 1,\n        "gradient_accumulation_steps": 16, "warmup_ratio": 0.05,\n        "lr_scheduler_type": "cosine", "weight_decay": 0.01,\n        "max_grad_norm": 0.3, "lora_r": 16, "lora_alpha": 32,\n        "lora_dropout": 0.05, "max_length": args.max_length,\n    }\n    metadata = {\n        "model": MODEL_ID, "model_commit": revision, "seed": args.seed,\n        "gpu": torch.cuda.get_device_name(0), "gpu_count_used": 1,\n        "train_sha256": hashlib.sha256(args.train.read_bytes()).hexdigest(),\n        "test_sha256": hashlib.sha256(args.test.read_bytes()).hexdigest(),\n        "train_rows": len(train_rows), "train_examples_used": len(training_indices),\n        "validation_examples": len(validation_indices), "test_rows": len(test_rows),\n        "excluded_long_examples": len(exclusions),\n        "train_token_lengths": {"min": min(lengths), "max": max(lengths), "mean": sum(lengths)/len(lengths)},\n        "settings": training_settings,\n        "versions": {p: importlib.metadata.version(p) for p in\n                     ["torch", "transformers", "peft", "accelerate", "bitsandbytes"]},\n        "protocol": "One epoch on training split only. Answer-only loss. Test evaluated once before and once after training, never used for selection. Fresh base and adapter share decoding and packages.",\n        "evaluation": {"max_new_tokens": 128, "do_sample": False,\n                       "invalid_answers_count_as_wrong": True, "prompt_example": build_prompt(test_rows[0])},\n    }\n    write_json(args.output / "metadata.json", metadata)\n    print(f"Training: {len(training_indices)}; validation: {len(validation_indices)}; exclusions: {len(exclusions)}", flush=True)\n    print("Measuring a fresh base-model reference with the same environment...", flush=True)\n    base_summary, base_predictions = evaluate_mcq(model, tokenizer, test_rows, args.output / "base")\n\n    # Reset stochastic state so training does not depend on evaluation RNG consumption.\n    set_seed(args.seed)\n    model.config.use_cache = False\n    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)\n    model = get_peft_model(model, LoraConfig(\n        r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",\n        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],\n    ))\n    model.print_trainable_parameters()\n    trainable, total = model.get_nb_trainable_parameters()\n    metadata.update(trainable_parameters=trainable, total_parameters=total)\n    write_json(args.output / "metadata.json", metadata)\n    training_args = TrainingArguments(\n        output_dir=str(args.output / "training"), num_train_epochs=1,\n        learning_rate=1e-4, per_device_train_batch_size=1, per_device_eval_batch_size=1,\n        gradient_accumulation_steps=16, gradient_checkpointing=True,\n        gradient_checkpointing_kwargs={"use_reentrant": False},\n        fp16=True, optim="paged_adamw_8bit", lr_scheduler_type="cosine",\n        warmup_ratio=0.05, weight_decay=0.01, max_grad_norm=0.3,\n        logging_steps=5, eval_strategy="epoch", save_strategy="no",\n        prediction_loss_only=True, report_to="none", seed=args.seed, data_seed=args.seed,\n        dataloader_num_workers=0, remove_unused_columns=False,\n    )\n    trainer = Trainer(\n        model=model, args=training_args,\n        train_dataset=[prepared[i] for i in training_indices],\n        eval_dataset=[prepared[i] for i in validation_indices],\n        processing_class=tokenizer,\n        data_collator=partial(collate_examples, pad_token_id=tokenizer.pad_token_id),\n    )\n    torch.cuda.reset_peak_memory_stats()\n    trained = trainer.train()\n    write_json(args.output / "training_metrics.json", {\n        **trained.metrics, "peak_gpu_allocated_gib": torch.cuda.max_memory_allocated()/1024**3,\n    })\n    write_json(args.output / "training_log.json", trainer.state.log_history)\n    adapter_path = args.output / "adapter"\n    model.save_pretrained(adapter_path, safe_serialization=True)\n    tokenizer.save_pretrained(adapter_path)\n\n    # Reload from disk before final scoring, verifying adapter portability.\n    del trainer, model\n    gc.collect()\n    torch.cuda.empty_cache()\n    base = AutoModelForCausalLM.from_pretrained(\n        MODEL_ID, revision=revision, quantization_config=quantization,\n        dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa",\n    )\n    model = PeftModel.from_pretrained(base, str(adapter_path), is_trainable=False)\n    tuned_summary, tuned_predictions = evaluate_mcq(model, tokenizer, test_rows, args.output / "finetuned")\n    comparison = {\n        "base": base_summary, "finetuned": tuned_summary,\n        "accuracy_change_percentage_points": 100*(tuned_summary["accuracy"]-base_summary["accuracy"]),\n        "relative_accuracy_change": (tuned_summary["accuracy"]-base_summary["accuracy"])/base_summary["accuracy"] if base_summary["accuracy"] else None,\n        "wrong_to_correct": sum(not a["correct"] and b["correct"] for a,b in zip(base_predictions,tuned_predictions)),\n        "correct_to_wrong": sum(a["correct"] and not b["correct"] for a,b in zip(base_predictions,tuned_predictions)),\n        "saved_adapter_reload_verified": True,\n    }\n    write_json(args.output / "comparison.json", comparison)\n    print(json.dumps(comparison, indent=2), flush=True)\n    print(f"Completed. Adapter and results: {args.output}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n',encoding='utf-8')


In [ ]:
# Exact current script: finetune_qwen_mcq.py
Path('finetune_qwen_mcq.py').write_text('"""Four-exposure MCQ QLoRA experiment; original assessment evaluator is unchanged."""\n\nimport argparse\nfrom collections import Counter\nfrom functools import partial\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport random\nimport re\n\n# Set before torch initializes in the training subprocess.\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\n\nfrom benchmark import build_prompt\nfrom finetune_qwen_coder import MODEL_ID, collate_examples, evaluate_mcq, read_csv, write_json\n\nLETTERS = "abcd"\nMCQ_FIELDS = ["class", "Question", *LETTERS, "Answer"]\n\n\ndef assistant_only_loss(model, inputs, return_outputs=False, num_items_in_batch=None):\n    """Exact masked causal loss without materializing prompt vocabulary logits.\n\n    Qwen\'s logits_to_keep slices hidden states before the vocabulary projection.\n    Keep the predecessor of each supervised label, including the turn ending.\n    This also works with padding and different answer positions within a batch.\n    """\n    import torch\n    labels = inputs["labels"]\n    targets = labels[:, 1:]\n    positions = torch.where((targets != -100).any(dim=0))[0]\n    if positions.numel() == 0:\n        raise ValueError("Batch has no supervised next-token targets")\n    forward_inputs = {k: v for k, v in inputs.items() if k != "labels"}\n    outputs = model(**forward_inputs, logits_to_keep=positions, use_cache=False)\n    selected_targets = targets.index_select(1, positions).to(outputs.logits.device)\n    loss = torch.nn.functional.cross_entropy(\n        outputs.logits.float().reshape(-1, outputs.logits.shape[-1]),\n        selected_targets.reshape(-1), ignore_index=-100,\n        reduction="sum" if num_items_in_batch is not None else "mean",\n    )\n    if num_items_in_batch is not None:\n        # Preserve Trainer\'s token normalization across gradient accumulation.\n        if isinstance(num_items_in_batch, torch.Tensor):\n            num_items_in_batch = num_items_in_batch.to(loss.device)\n        loss = loss / num_items_in_batch\n    return (loss, outputs) if return_outputs else loss\n\n\ndef load_split(path, split):\n    rows = read_csv(path, MCQ_FIELDS + ["source_row", "split"])\n    ids = set()\n    for row in rows:\n        if row["split"] != split:\n            raise ValueError(f"Wrong split in {path}")\n        try:\n            source_id = int(row["source_row"])\n        except ValueError as exc:\n            raise ValueError("Invalid source_row") from exc\n        if source_id < 0 or source_id in ids:\n            raise ValueError("Invalid or repeated source_row within a split")\n        ids.add(source_id)\n        row["Answer"] = row["Answer"].strip().lower()\n        if row["Answer"] not in list(LETTERS):\n            raise ValueError("Invalid answer letter")\n        if len({row[letter].strip() for letter in LETTERS}) != 4:\n            raise ValueError("Duplicate MCQ options")\n    return rows\n\n\ndef check_split_isolation(train, validation):\n    if {r["source_row"] for r in train} & {r["source_row"] for r in validation}:\n        raise ValueError("Source ids overlap between train and validation")\n    key = lambda r: re.sub(r"\\s+", " ", r["Question"].strip()).casefold()\n    if {key(r) for r in train} & {key(r) for r in validation}:\n        raise ValueError("Identical normalized questions cross splits")\n\n\ndef rotate_question(row, seed=42):\n    """Shuffle once per stable source id, then rotate all four options together."""\n    order = list(LETTERS)\n    seed_bytes = f"mcq-rotation-v1:{seed}:{row[\'source_row\']}".encode()\n    random.Random(int.from_bytes(hashlib.sha256(seed_bytes).digest(), "big")).shuffle(order)\n    result = []\n    for rotation in range(4):\n        permutation = order[rotation:] + order[:rotation]\n        mapping = dict(zip(LETTERS, permutation))  # new letter -> original letter\n        answer = next(new for new, old in mapping.items() if old == row["Answer"])\n        rotated = {**row, **{new: row[old] for new, old in mapping.items()}, "Answer": answer}\n        result.append((rotated, {"source_row": int(row["source_row"]), "split": row["split"],\n                                  "rotation": rotation, "new_to_original": mapping,\n                                  "original_answer": row["Answer"], "answer": answer}))\n    return result\n\n\ndef tokenize_mcq(row, tokenizer, max_length):\n    messages = [{"role": "user", "content": build_prompt(row)}]\n    prefix = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=False)\n    full = tokenizer.apply_chat_template(\n        messages + [{"role": "assistant", "content": "Answer:" + row["Answer"]}],\n        tokenize=True, add_generation_prompt=False, return_dict=False,\n    )\n    if full[:len(prefix)] != prefix:\n        raise ValueError("Chat-template prefix mismatch; refusing incorrect loss masking")\n    if len(full) <= len(prefix):\n        raise ValueError("No assistant target tokens")\n    if len(full) > max_length:\n        return None, len(full)\n    return {"input_ids": full, "attention_mask": [1] * len(full),\n            "labels": [-100] * len(prefix) + full[len(prefix):]}, len(full)\n\n\ndef prepare_split(rows, tokenizer, max_length, seed, training):\n    prepared, admitted, maps, exclusions = [], [], [], []\n    for row in rows:\n        variants = rotate_question(row, seed)\n        encoded = [tokenize_mcq(v, tokenizer, max_length) for v, _ in variants]\n        fixed, fixed_length = tokenize_mcq(row, tokenizer, max_length)\n        lengths = [length for _, length in encoded]\n        # Use the same length-eligible validation cohort for fixed and rotated metrics.\n        if any(example is None for example, _ in encoded) or fixed is None:\n            exclusions.append({"source_row": int(row["source_row"]), "split": row["split"],\n                               "rotation_lengths": lengths, "fixed_length": fixed_length,\n                               "reason": "Entire question group exceeds max_length; no truncation"})\n            continue\n        admitted.append(row)\n        if training:\n            prepared.extend(example for example, _ in encoded)\n        else:\n            prepared.append(fixed)\n        for (_, mapping), length in zip(variants, lengths):\n            maps.append({**mapping, "tokens": length})\n    if not admitted:\n        raise ValueError("No questions fit max_length in a split")\n    if training:\n        random.Random(seed).shuffle(prepared)\n        assert Counter(m["answer"] for m in maps) == {l: len(admitted) for l in LETTERS}\n    return prepared, admitted, maps, exclusions\n\n\ndef smoke_training_rows(rows, tokenizer, max_length, seed):\n    """Include the longest fitting question so smoke tests exercise peak lengths."""\n    _, admitted, maps, _ = prepare_split(rows, tokenizer, max_length, seed, False)\n    longest_id = max(maps, key=lambda m: m["tokens"])["source_row"]\n    longest = next(r for r in admitted if int(r["source_row"]) == longest_id)\n    return [longest] + [r for r in admitted if r is not longest][:7]\n\n\ndef rotation_diagnostics(rows, predictions, seed):\n    if len(predictions) != 4 * len(rows):\n        raise ValueError("Incomplete rotated predictions")\n    details, by_position = [], {l: {"correct": 0, "questions": 0} for l in LETTERS}\n    for index, row in enumerate(rows):\n        selected = []\n        correct = []\n        for offset, (_, mapping) in enumerate(rotate_question(row, seed)):\n            prediction = predictions[4 * index + offset]\n            letter = prediction["prediction"]\n            selected.append(mapping["new_to_original"].get(letter))\n            correct.append(prediction["correct"])\n            stat = by_position[mapping["answer"]]\n            stat["questions"] += 1\n            stat["correct"] += prediction["correct"]\n        details.append({"source_row": int(row["source_row"]),\n                        "selected_original_letters": selected,\n                        "consistent": None not in selected and len(set(selected)) == 1,\n                        "all_four_correct": all(correct), "correct_rotations": sum(correct)})\n    for stat in by_position.values():\n        stat["accuracy"] = stat["correct"] / stat["questions"]\n    return {"questions": len(rows), "accuracy_by_correct_position": by_position,\n            "consistent_answer_text_rate": sum(r["consistent"] for r in details) / len(rows),\n            "all_four_correct_rate": sum(r["all_four_correct"] for r in details) / len(rows),\n            "note": "Invalid responses are inconsistent. Consistency alone does not imply correctness.",\n            "per_question": details}\n\n\ndef score(model, tokenizer, validation, test, output, seed):\n    # Do not build a graph or retain KV caching during loss evaluation in Trainer.\n    fixed, _ = evaluate_mcq(model, tokenizer, validation, output / "validation")\n    test_summary, test_predictions = evaluate_mcq(model, tokenizer, test, output / "test")\n    rotated = [v for row in validation for v, _ in rotate_question(row, seed)]\n    rotated_summary, predictions = evaluate_mcq(model, tokenizer, rotated, output / "validation_rotated")\n    diagnostic = rotation_diagnostics(validation, predictions, seed)\n    diagnostic["aggregate"] = rotated_summary\n    write_json(output / "rotation_diagnostics.json", diagnostic)\n    return {"validation": fixed, "test": test_summary, "rotation": diagnostic}, test_predictions\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--train", type=Path, default=Path("data/mcq/generated-mcqs-final/train.csv"))\n    parser.add_argument("--validation", type=Path, default=Path("data/mcq/generated-mcqs-final/validation.csv"))\n    parser.add_argument("--test", type=Path, default=Path("TestData.csv"))\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--max-length", type=int, default=4096)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--prepare-only", action="store_true", help="Tokenizer preflight; no model load or GPU required")\n    parser.add_argument("--smoke-test", action="store_true", help="8 train questions, 4 validation, 2 test; not benchmark results")\n    args = parser.parse_args(argv)\n    if args.max_length < 64:\n        parser.error("max-length must be at least 64")\n    train = load_split(args.train, "train")\n    validation = load_split(args.validation, "validation")\n    check_split_isolation(train, validation)\n    total_train, total_validation = len(train), len(validation)\n    args.output.mkdir(parents=True, exist_ok=False)\n    from transformers import AutoTokenizer, set_seed\n    set_seed(args.seed)\n    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)\n    if tokenizer.pad_token_id is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    tokenizer.padding_side = "right"\n    if args.smoke_test:\n        train = smoke_training_rows(train, tokenizer, args.max_length, args.seed)\n        validation = validation[:4]\n    train_data, train, train_maps, train_exclusions = prepare_split(train, tokenizer, args.max_length, args.seed, True)\n    val_data, validation, val_maps, val_exclusions = prepare_split(validation, tokenizer, args.max_length, args.seed, False)\n    metadata = {\n        "model": MODEL_ID, "seed": args.seed, "smoke_test": args.smoke_test,\n        "source_counts": {"train": total_train, "validation": total_validation},\n        "admitted_questions": {"train": len(train), "validation": len(validation)},\n        "training_presentations": len(train_data), "train_label_counts": dict(Counter(m["answer"] for m in train_maps)),\n        "training_source_rows": [int(r["source_row"]) for r in train],\n        "input_sha256": {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in\n                         [("train", args.train), ("validation", args.validation)]},\n        "protocol": "One epoch of four rotations per question (four original-question exposures). Fixed validation plus rotation diagnostics. Test labels never enter training. Not the original single-exposure assessment budget.",\n        "settings": {"rotations": 4, "max_length": args.max_length, "num_train_epochs": 1,\n                     "learning_rate": 1e-4, "batch_size": 1, "gradient_accumulation_steps": 16,\n                     "lora_r": 16, "lora_alpha": 32, "lora_dropout": 0.05,\n                     "target_format": "Answer:<letter>", "loss": "assistant response only",\n                     "loss_implementation": "selected causal logits before vocabulary projection",\n                     "allocator": os.environ.get("PYTORCH_CUDA_ALLOC_CONF")},\n        "evaluation": {"max_new_tokens": 128, "do_sample": False, "invalid_answers_count_as_wrong": True,\n                       "prompt_example": build_prompt(validation[0]),\n                       "validation_cohort": "All four orders and original order must fit max_length; full exclusions in length_exclusions.json"},\n    }\n    write_json(args.output / "metadata.json", metadata)\n    write_json(args.output / "length_exclusions.json", train_exclusions + val_exclusions)\n    with (args.output / "rotation_map.jsonl").open("w", encoding="utf-8") as handle:\n        for mapping in train_maps + val_maps:\n            handle.write(json.dumps(mapping) + "\\n")\n    print(f"Admitted: {len(train)} train questions -> {len(train_data)} presentations; {len(validation)} validation questions. Excluded groups: {len(train_exclusions) + len(val_exclusions)}.", flush=True)\n    if args.prepare_only:\n        print("Preflight complete; no model was loaded or trained.", flush=True)\n        return\n\n    import torch\n    from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training\n    from transformers import AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments\n    if not torch.cuda.is_available():\n        raise RuntimeError("Enable a Colab CUDA GPU. Tokenizer preflight is saved.")\n    test = read_csv(args.test, MCQ_FIELDS)\n    if any(r["Answer"].strip().lower() not in list(LETTERS) for r in test):\n        raise ValueError("Invalid test answer label")\n    if args.smoke_test:\n        test = test[:2]\n    # Include the assistant target in context checks; never truncate evaluation silently.\n    if any(tokenize_mcq(r, tokenizer, args.max_length)[0] is None for r in test):\n        raise ValueError("A test prompt exceeds max_length; increase the configured context before running")\n    quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",\n                                   bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)\n    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=quantization,\n                                                dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa")\n    revision = getattr(model.config, "_commit_hash", None)\n    metadata.update(model_commit=revision, gpu=torch.cuda.get_device_name(0), gpu_count_used=1,\n                    test_rows=len(test), test_sha256=hashlib.sha256(args.test.read_bytes()).hexdigest(),\n                    versions={p: importlib.metadata.version(p) for p in\n                              ["torch", "transformers", "peft", "accelerate", "bitsandbytes"]})\n    write_json(args.output / "metadata.json", metadata)\n    base_summary, base_predictions = score(model, tokenizer, validation, test, args.output / "base", args.seed)\n    set_seed(args.seed)\n    model.train()\n    model.config.use_cache = False\n    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)\n    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,\n        bias="none", task_type="CAUSAL_LM",\n        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))\n    model.print_trainable_parameters()\n    trainable, total = model.get_nb_trainable_parameters()\n    metadata.update(trainable_parameters=trainable, total_parameters=total)\n    write_json(args.output / "metadata.json", metadata)\n    training_args = TrainingArguments(\n        output_dir=str(args.output / "training"), num_train_epochs=1, learning_rate=1e-4,\n        per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=16,\n        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},\n        fp16=True, optim="paged_adamw_8bit", lr_scheduler_type="cosine", warmup_ratio=0.05,\n        weight_decay=0.01, max_grad_norm=0.3, logging_steps=1 if args.smoke_test else 5,\n        eval_strategy="epoch", save_strategy="no", prediction_loss_only=True, report_to="none",\n        seed=args.seed, data_seed=args.seed, dataloader_num_workers=0, remove_unused_columns=False,\n    )\n    class MCQTrainer(Trainer):\n        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):\n            return assistant_only_loss(model, inputs, return_outputs, num_items_in_batch)\n\n    trainer = MCQTrainer(model=model, args=training_args, train_dataset=train_data, eval_dataset=val_data,\n                      processing_class=tokenizer, data_collator=partial(collate_examples, pad_token_id=tokenizer.pad_token_id))\n    trainer.model_accepts_loss_kwargs = True\n    gc.collect()\n    torch.cuda.empty_cache()\n    torch.cuda.reset_peak_memory_stats()\n    trained = trainer.train()\n    write_json(args.output / "training_metrics.json", {**trained.metrics,\n               "peak_gpu_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3})\n    write_json(args.output / "training_log.json", trainer.state.log_history)\n    adapter = args.output / "adapter"\n    model.save_pretrained(adapter, safe_serialization=True)\n    tokenizer.save_pretrained(adapter)\n    del trainer, model\n    gc.collect()\n    torch.cuda.empty_cache()\n    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=revision, quantization_config=quantization,\n                                                dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa")\n    model = PeftModel.from_pretrained(model, str(adapter), is_trainable=False)\n    tokenizer = AutoTokenizer.from_pretrained(adapter)\n    tuned_summary, tuned_predictions = score(model, tokenizer, validation, test, args.output / "finetuned", args.seed)\n    comparison = {"base": base_summary, "finetuned": tuned_summary, "smoke_test": args.smoke_test,\n        "test_accuracy_change_percentage_points": 100 * (tuned_summary["test"]["accuracy"] - base_summary["test"]["accuracy"]),\n        "wrong_to_correct": sum(not a["correct"] and b["correct"] for a, b in zip(base_predictions, tuned_predictions)),\n        "correct_to_wrong": sum(a["correct"] and not b["correct"] for a, b in zip(base_predictions, tuned_predictions)),\n        "saved_adapter_reload_verified": True}\n    write_json(args.output / "comparison.json", comparison)\n    print(f"Completed. Reloaded adapter and results: {args.output}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n',encoding='utf-8')


In [ ]:
assert 'logits_to_keep=positions' in Path('finetune_qwen_mcq.py').read_text()
results=[]
for limit in [4096,6144]:
    folder=TEST_ROOT/str(limit)
    command=[sys.executable,'-u','finetune_qwen_mcq.py','--train',str(DATA_DIR/'train.csv'),
             '--validation',str(DATA_DIR/'validation.csv'),'--test',str(DATA_DIR/'TestData.csv'),
             '--output',str(folder),'--max-length',str(limit),'--seed','42','--smoke-test']
    print(f'\n===== Longest-question GPU smoke test: {limit} tokens =====',flush=True)
    with (TEST_ROOT/f'{limit}-console.log').open('w',encoding='utf-8') as log:
        with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
            for line in process.stdout:
                print(line,end='',flush=True);log.write(line);log.flush()
            code=process.wait()
    row={'max_length':limit,'exit_code':code,'passed':False}
    if code==0:
        comparison=json.loads((folder/'comparison.json').read_text())
        metrics=json.loads((folder/'training_metrics.json').read_text())
        metadata=json.loads((folder/'metadata.json').read_text())
        row.update(passed=bool(comparison['saved_adapter_reload_verified']),
                   peak_gpu_allocated_gib=metrics['peak_gpu_allocated_gib'],
                   training_source_rows=metadata['training_source_rows'],
                   presentations=metadata['training_presentations'])
    results.append(row)
    (TEST_ROOT/'memory-test-results.json').write_text(json.dumps(results,indent=2),encoding='utf-8')
    print('TEST RESULT:',json.dumps(row),flush=True)
    if not row['passed']:
        print('Stopped; the larger context was not attempted. Inspect the console log above.',flush=True)
        break
if len(results)==2 and all(r['passed'] for r in results):
    print('BOTH GPU SMOKE TESTS PASSED. 6144 is a candidate for the full run; no full run has been started.',flush=True)
print('Saved evidence:',TEST_ROOT)
